# 04 - Dataset Quality Check

**Goal:** Detect corrupted images, empty folders, duplicate files, and class imbalance in the PlantVillage dataset.

| Info | Value |
|------|-------|
| Dataset | PlantVillage |
| Task | Multi-Crop Disease Detection |
| Phase | 4 — Dataset Quality Check |

In [ ]:
import os
import hashlib
from PIL import Image
from collections import defaultdict

DATASET_PATH = "../dataset/PlantVillage"

In [ ]:
# ✅ 1. Check for Empty Folders
print("=" * 50)
print("1. Empty Folder Check")
print("=" * 50)

empty_folders = []

for class_name in os.listdir(DATASET_PATH):
    class_path = os.path.join(DATASET_PATH, class_name)
    if os.path.isdir(class_path):
        images = [f for f in os.listdir(class_path) if f.lower().endswith((".png", ".jpg", ".jpeg"))]
        if len(images) == 0:
            empty_folders.append(class_name)

if empty_folders:
    print(f"⚠️  Empty folders found: {empty_folders}")
else:
    print("✅ No empty folders found!")

In [ ]:
# ✅ 2. Corrupted Image Detection
print("=" * 50)
print("2. Corrupted Image Detection")
print("=" * 50)

corrupted_images = []

for class_name in os.listdir(DATASET_PATH):
    class_path = os.path.join(DATASET_PATH, class_name)
    if not os.path.isdir(class_path):
        continue
    for image_name in os.listdir(class_path):
        image_path = os.path.join(class_path, image_name)
        try:
            img = Image.open(image_path)
            img.verify()  # Verify integrity
        except Exception as e:
            corrupted_images.append(image_path)

if corrupted_images:
    print(f"⚠️  Corrupted images found: {len(corrupted_images)}")
    for p in corrupted_images:
        print("  -", p)
else:
    print("✅ No corrupted images found!")

In [ ]:
# ✅ 3. Duplicate Image Detection (using MD5 hash)
print("=" * 50)
print("3. Duplicate Image Detection")
print("=" * 50)

hash_map = defaultdict(list)

for class_name in os.listdir(DATASET_PATH):
    class_path = os.path.join(DATASET_PATH, class_name)
    if not os.path.isdir(class_path):
        continue
    for image_name in os.listdir(class_path):
        image_path = os.path.join(class_path, image_name)
        try:
            with open(image_path, "rb") as f:
                file_hash = hashlib.md5(f.read()).hexdigest()
            hash_map[file_hash].append(image_path)
        except:
            pass

duplicates = {k: v for k, v in hash_map.items() if len(v) > 1}

if duplicates:
    total_dupes = sum(len(v) - 1 for v in duplicates.values())
    print(f"⚠️  Duplicate images found: {total_dupes} duplicates in {len(duplicates)} groups")
else:
    print("✅ No duplicate images found!")

In [ ]:
# ✅ 4. Class Imbalance Check
print("=" * 50)
print("4. Class Imbalance Check")
print("=" * 50)

class_counts = {}

for class_name in sorted(os.listdir(DATASET_PATH)):
    class_path = os.path.join(DATASET_PATH, class_name)
    if os.path.isdir(class_path):
        images = [f for f in os.listdir(class_path) if f.lower().endswith((".png", ".jpg", ".jpeg"))]
        class_counts[class_name] = len(images)

min_class = min(class_counts, key=class_counts.get)
max_class = max(class_counts, key=class_counts.get)

print(f"\n🔺 Largest class  : {max_class} ({class_counts[max_class]} images)")
print(f"🔻 Smallest class : {min_class} ({class_counts[min_class]} images)")
print(f"\n📊 Imbalance Ratio : {class_counts[max_class] / class_counts[min_class]:.1f}x")

if class_counts[max_class] / class_counts[min_class] > 5:
    print("\n⚠️  High class imbalance! Consider using class_weight in training.")
else:
    print("\n✅ Class imbalance is acceptable.")